# Heston-Arb Live Loop — Google Colab

**Setup order:** run cells 1→5 once, then Cell 6 runs the loop.

State (positions, Heston params, tick log) is written to Google Drive so it
survives session restarts. The loop resumes automatically from the last tick.

> **Runtime:** Runtime → Change runtime type → T4 GPU (10–30× faster JAX)

In [2]:
# ── Cell 1: GPU check ──────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'No GPU — switch runtime to T4')

GPU: Tesla T4, 15360 MiB


In [3]:
# ── Cell 2: Install dependencies ───────────────────────────────────────────────
# Installs take ~3 min on first run. Skip on subsequent runs if kernel is warm.
!pip install -q alpaca-py cvxpy optax diffrax scipy financepy

# JAX with CUDA (GPU). If GPU is unavailable, falls back to CPU automatically.
!pip install -q -U "jax[cuda12]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
!pip install -q numpyro

import jax
print('JAX devices:', jax.devices())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.5/122.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.7/199.7 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.9/793.9 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.6/77.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.5/85.5 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.4/174.4 MB 6.8 MB/s eta 0:00:00
JAX devices: [CudaDevice(id=0)]


In [4]:
# Cell 3: Clone / update repo
# Requires GITHUB_TOKEN in Colab Secrets (Key icon, left sidebar).

import os, subprocess, sys, shutil
from google.colab import userdata

REPO = '/content/heston-arb'
token = userdata.get('GITHUB_TOKEN')
clone_url = f'https://{token}@github.com/AliAlpOezer/heston-arb.git'

# Check for a valid repo, not just the directory (failed clone leaves empty dir)
is_valid_repo = os.path.exists(os.path.join(REPO, '.git'))

if not is_valid_repo:
    if os.path.exists(REPO):
        shutil.rmtree(REPO)  # remove stale/partial clone
    r = subprocess.run(['git', 'clone', clone_url, REPO], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'Clone failed: {r.stderr.strip()}')
    print('Cloned successfully.')
else:
    r = subprocess.run(['git', '-C', REPO, 'pull'], capture_output=True, text=True)
    print('Pulled:', r.stdout.strip() or r.stderr.strip())

if REPO not in sys.path:
    sys.path.insert(0, REPO)

print('Files in repo:', os.listdir(REPO))
print('Repo ready at', REPO)


Cloned successfully.
Files in repo: ['config.py', 'data', 'evals', '.git', '.gitignore', 'docs', 'signals', 'backtest', 'README.md', '.env.example', 'CLAUDE.md', 'risk', 'trading', 'colab.ipynb', 'calibration', 'requirements.txt']
Repo ready at /content/heston-arb


In [5]:
# ── Cell 4: Mount Google Drive + configure state paths ─────────────────────────
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# All persistent files go here — survives session restarts
STATE_DIR = Path('/content/drive/MyDrive/heston-arb-live')
STATE_DIR.mkdir(parents=True, exist_ok=True)

STATE_PATH     = STATE_DIR / 'live_state.json'
KILL_LOG_PATH  = STATE_DIR / 'kill_log.json'
KILL_FLAG_PATH = STATE_DIR / 'kill_flag.json'
TICK_LOG_PATH  = STATE_DIR / 'tick_log.jsonl'   # one line per tick

print('State directory:', STATE_DIR)
print('Existing files:', [f.name for f in STATE_DIR.iterdir()] if STATE_DIR.exists() else '(empty)')

Mounted at /content/drive
State directory: /content/drive/MyDrive/heston-arb-live
Existing files: ['tick_log.jsonl', 'live_state.json', 'kill_log.json', 'kill_flag.json']


In [6]:
# Cell 5: API keys
# Add secrets via the Key icon in the left sidebar:
#   ALPACA_API_KEY, ALPACA_SECRET_KEY, FRED_API_KEY (optional)
# Toggle "Notebook access" ON for each secret.

import os, json
from google.colab import userdata
from urllib.request import urlopen, Request
from urllib.error import HTTPError

os.environ['ALPACA_API_KEY']    = userdata.get('ALPACA_API_KEY')
os.environ['ALPACA_SECRET_KEY'] = userdata.get('ALPACA_SECRET_KEY')
fred = userdata.get('FRED_API_KEY') or ''
if fred:
    os.environ['FRED_API_KEY'] = fred

req = Request(
    'https://paper-api.alpaca.markets/v2/account',
    headers={
        'APCA-API-KEY-ID':     os.environ['ALPACA_API_KEY'],
        'APCA-API-SECRET-KEY': os.environ['ALPACA_SECRET_KEY'],
    },
)
try:
    with urlopen(req, timeout=10) as r:
        acct = json.loads(r.read())
    print(f'Alpaca OK -- equity=${acct["equity"]}  buying_power=${acct["buying_power"]}')
except HTTPError as e:
    print(f'Alpaca auth FAILED ({e.code}) -- check your keys')


Alpaca OK -- equity=$100000  buying_power=$400000


In [7]:
# Cell 6: Run live loop
import sys, os

REPO = '/content/heston-arb'

# --- diagnostics ---
print("REPO exists:",       os.path.exists(REPO))
print("trading/ exists:",   os.path.exists(REPO + '/trading'))
print("__init__.py:",       os.path.exists(REPO + '/trading/__init__.py'))
print("loop.py:",           os.path.exists(REPO + '/trading/loop.py'))
print("sys.path[0]:",       sys.path[0])

# Insert path
if REPO not in sys.path:
    sys.path.insert(0, REPO)

# Clear stale cached failures (Python caches failed imports in sys.modules)
for _k in [k for k in sys.modules if k == "trading" or k.startswith("trading.")]:
    del sys.modules[_k]

from trading.loop import run_live
from trading.broker import AlpacaBroker
from data.alpaca_loader import AlpacaLoader

run_live(
    ticker           = 'SPY',
    broker           = AlpacaBroker(paper=True),
    loader           = AlpacaLoader(call_only=False),
    interval_minutes = 5,
    dry_run          = False,
    verbose          = True,
    state_path       = STATE_PATH,
    kill_log_path    = KILL_LOG_PATH,
    kill_flag_path   = KILL_FLAG_PATH,
    tick_log_path    = TICK_LOG_PATH,
)


REPO exists: True
trading/ exists: True
__init__.py: True
loop.py: True
sys.path[0]: /content/heston-arb


ModuleNotFoundError: No module named 'numpyro'

In [ ]:
# ── Cell 7: Analyse tick log (run any time in a separate cell) ─────────────────
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_json(TICK_LOG_PATH, lines=True)
df['time'] = pd.to_datetime(df['time'])
df = df.set_index('time')

print(df[['spot', 'rmse', 'n_signals_filtered', 'n_entered', 'session_pnl']].tail(20).to_string())

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
df['rmse'].plot(ax=axes[0], title='Calibration RMSE')
df['n_signals_filtered'].plot(ax=axes[1], title='Filtered signals per tick')
df['session_pnl'].plot(ax=axes[2], title='Cumulative session P&L')
plt.tight_layout()
plt.show()